<a href="https://colab.research.google.com/github/QuantLet/CBDC/blob/main/China%20Test%20Market/Heatmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install akshare, geopandas, and geopy if not already installed
!pip install akshare
!pip install geopandas
!pip install geopy
# Install selenium and webdriver-manager
!pip install selenium --quiet
!pip install webdriver-manager --quiet

# Add Google Chrome repository and install google-chrome-stable
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" | sudo tee /etc/apt/sources.list.d/google-chrome.list
!sudo apt-get update
!sudo apt-get install google-chrome-stable --yes

import akshare as ak
import geopandas as gpd
import matplotlib.pyplot as plt
from geopy.geocoders import Nominatim
from shapely.geometry import Point
import folium
import pandas as pd

import os
import time
from PIL import Image
import imageio
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import shutil

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 71.9 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=ea37bbf6092621829c734a1cb3aa90dc9f82c26879336d164b659dea9d7536bb
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.9 MB/s eta 0:00:00
OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-

In [3]:
print([attr for attr in dir(ak) if 'macro' in attr])

['macro_australia_bank_rate', 'macro_australia_cpi_quarterly', 'macro_australia_cpi_yearly', 'macro_australia_ppi_quarterly', 'macro_australia_retail_rate_monthly', 'macro_australia_trade', 'macro_australia_unemployment_rate', 'macro_bank_australia_interest_rate', 'macro_bank_brazil_interest_rate', 'macro_bank_china_interest_rate', 'macro_bank_english_interest_rate', 'macro_bank_euro_interest_rate', 'macro_bank_india_interest_rate', 'macro_bank_japan_interest_rate', 'macro_bank_newzealand_interest_rate', 'macro_bank_russia_interest_rate', 'macro_bank_switzerland_interest_rate', 'macro_bank_usa_interest_rate', 'macro_canada_bank_rate', 'macro_canada_core_cpi_monthly', 'macro_canada_core_cpi_yearly', 'macro_canada_cpi_monthly', 'macro_canada_cpi_yearly', 'macro_canada_gdp_monthly', 'macro_canada_new_house_rate', 'macro_canada_retail_rate_monthly', 'macro_canada_trade', 'macro_canada_unemployment_rate', 'macro_china_agricultural_index', 'macro_china_agricultural_product', 'macro_china_au_

In [4]:


# Download Natural Earth GeoJSON if not already present
# Using a direct GeoJSON link from GitHub for stability
!rm -f ne_10m_admin_1_states_provinces.geojson
!wget https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_1_states_provinces.geojson

# REAL DATA 1: China provincial GDP & retail sales from NBS via akshare
macro_df = ak.macro_china_consumer_goods_retail()  # Real NBS monthly retail sales
print(macro_df.tail())

# REAL DATA 2: Geospatial plotting of actual pilot cities on real China map
# Natural Earth GeoJSON (public domain): https://github.com/nvkelso/natural-earth-vector
china = gpd.read_file("ne_10m_admin_1_states_provinces.geojson")
china = china[china['admin'] == 'China']

pilots = [
    {"name": "Shenzhen", "lat": 22.5431, "lon": 114.0579, "wave": 1, "type": "Key City", "note": "Red envelope trials; HK proximity"},
    {"name": "Chengdu", "lat": 30.5728, "lon": 104.0668, "wave": 1, "type": "Key City", "note": "Inland inclusion test"},
    {"name": "Suzhou", "lat": 31.2989, "lon": 120.5853, "wave": 1, "type": "Key City", "note": "SOE salary payments"},
    {"name": "Xiong'an", "lat": 38.9903, "lon": 115.9023, "wave": 1, "type": "Key City", "note": "Smart city; blockchain R&D"},
    {"name": "Shanghai", "lat": 31.2304, "lon": 121.4737, "wave": 2, "type": "Municipality", "note": "Financial center"},
    {"name": "Haikou", "lat": 20.0440, "lon": 110.1999, "wave": 2, "type": "Border/Trade", "note": "Free trade port"},
    {"name": "Changsha", "lat": 28.2280, "lon": 112.9388, "wave": 2, "type": "Key City", "note": "Consumer retail"},
    {"name": "Xi'an", "lat": 34.3416, "lon": 108.9398, "wave": 2, "type": "Key City", "note": "Belt & Road hub"},
    {"name": "Qingdao", "lat": 36.0671, "lon": 120.3826, "wave": 2, "type": "Key City", "note": "Port city"},
    {"name": "Dalian", "lat": 38.9140, "lon": 121.6147, "wave": 2, "type": "Key City", "note": "Northeast port"},
    {"name": "Beijing", "lat": 39.9042, "lon": 116.4074, "wave": 2, "type": "Municipality", "note": "Winter Olympics; foreigner access"},
    {"name": "Tianjin", "lat": 39.0842, "lon": 117.2010, "wave": 3, "type": "Municipality", "note": "Northern port"},
    {"name": "Chongqing", "lat": 29.5630, "lon": 106.5516, "wave": 3, "type": "Municipality", "note": "Western hub"},
    {"name": "Guangzhou", "lat": 23.1291, "lon": 113.2644, "wave": 3, "type": "Key City", "note": "Greater Bay Area"},
    {"name": "Fuzhou", "lat": 26.0745, "lon": 119.2965, "wave": 3, "type": "Key City", "note": "Cross-strait trade"},
    {"name": "Xiamen", "lat": 24.4798, "lon": 118.0894, "wave": 3, "type": "Key City", "note": "SEZ; Taiwan-facing"},
    {"name": "Hangzhou", "lat": 30.2741, "lon": 120.1551, "wave": 3, "type": "Key City", "note": "Alipay hometown"},
    {"name": "Ningbo", "lat": 29.8683, "lon": 121.5440, "wave": 3, "type": "Key City", "note": "Port + logistics"},
    {"name": "Wenzhou", "lat": 28.0009, "lon": 120.7022, "wave": 3, "type": "Key City", "note": "SME finance"},
    {"name": "Huzhou", "lat": 30.8925, "lon": 120.0880, "wave": 3, "type": "Key City", "note": "Green finance"},
    {"name": "Shaoxing", "lat": 30.0023, "lon": 120.5790, "wave": 3, "type": "Key City", "note": "Textile manufacturing"},
    {"name": "Jinhua", "lat": 29.0781, "lon": 119.6472, "wave": 3, "type": "Key City", "note": "E-commerce logistics"},
    {"name": "Shijiazhuang", "lat": 38.0428, "lon": 114.5149, "wave": 4, "type": "Province Rep", "note": "Hebei corridor"},
    {"name": "Nanjing", "lat": 32.0603, "lon": 118.7969, "wave": 4, "type": "Province Rep", "note": "Jiangsu capital"},
    {"name": "Jinan", "lat": 36.6512, "lon": 117.1201, "wave": 4, "type": "Province Rep", "note": "Shandong capital"},
    {"name": "Nanning", "lat": 22.8170, "lon": 108.3665, "wave": 4, "type": "Border/Trade", "note": "ASEAN-facing"},
    {"name": "Fangchenggang", "lat": 21.6867, "lon": 108.3547, "wave": 4, "type": "Border/Trade", "note": "Vietnam border"},
    {"name": "Kunming", "lat": 25.0389, "lon": 102.7183, "wave": 4, "type": "Border/Trade", "note": "SE Asia gateway"},
    {"name": "Jinghong", "lat": 21.9555, "lon": 100.4598, "wave": 4, "type": "Border/Trade", "note": "Laos/Thailand border"},
]

df = pd.DataFrame(pilots)
wave_colors = {1: '#e74c3c', 2: '#3498db', 3: '#2ecc71', 4: '#9b59b6'}
wave_names = {1: 'Wave 1: Apr 2020', 2: 'Wave 2: Oct 2020', 3: 'Wave 3: Apr 2022', 4: 'Wave 4: Dec 2022'}
radius_map = {'Municipality': 12, 'Key City': 10, 'Province Rep': 8, 'Border/Trade': 9}

m = folium.Map(location=[32.0, 110.0], zoom_start=4, tiles='CartoDB positron')
feature_groups = {}
for wave in [1, 2, 3, 4]:
    fg = folium.FeatureGroup(name=wave_names[wave])
    feature_groups[wave] = fg
    m.add_child(fg)

for _, row in df.iterrows():
    popup = f"<b>{row['name']}</b><br>Wave: {row['wave']}<br>Type: {row['type']}<br>{row['note']}"
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius_map.get(row['type'], 8),
        color=wave_colors[row['wave']], fill=True, fillColor=wave_colors[row['wave']],
        fillOpacity=0.7, weight=2, popup=folium.Popup(popup, max_width=250)
    ).add_to(feature_groups[row['wave']])

folium.LayerControl(collapsed=False).add_to(m)
m.save("ecny_pilot_live_map.html")
print("Map saved to ecny_pilot_live_map.html")

--2026-08-04 13:03:40--  https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_1_states_provinces.geojson
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40726851 (39M) [text/plain]
Saving to: ‘ne_10m_admin_1_states_provinces.geojson’

ne_10m_admin_1_stat 100%[===================>]  38.84M  67.8MB/s    in 0.6s    

2026-08-04 13:03:42 (67.8 MB/s) - ‘ne_10m_admin_1_states_provinces.geojson’ saved [40726851/40726851]

            月份      当月  同比增长      环比增长       累计  累计-同比增长
202  2008年05月份  8703.5  21.6  6.896340  42400.7     21.1
203  2008年04月份  8142.0  22.0  0.231436  33697.2     21.0
204  2008年03月份  8123.2  21.5 -2.770895  25555.2     20.6
205  2008年02月份  8354.7  19.1 -7.960517  17432.0     20.2
206  2008年01月份  9077.3  21

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager # Import ChromeDriverManager
import shutil
import os
import time

# Setup headless Chrome for rendering Folium maps
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.binary_location = '/usr/bin/google-chrome'

# Use ChromeDriverManager to automatically download and install the correct chromedriver
try:
    chromedriver_path = ChromeDriverManager().install()
except Exception as e:
    print(f"Error installing chromedriver: {e}")
    # Fallback to shutil.which if webdriver_manager fails
    chromedriver_path = shutil.which("chromedriver")

# Fallback if chromedriver_path is still None
if chromedriver_path is None:
    if os.path.exists("/usr/bin/chromedriver"):
        chromedriver_path = "/usr/bin/chromedriver"
    else:
        raise Exception("Chromedriver not found. Please ensure google-chrome-stable is installed and check your internet connection for webdriver_manager.")

service = Service(executable_path=chromedriver_path)
driver = webdriver.Chrome(service=service, options=chrome_options)

def save_folium_map_as_png(folium_map, output_filename, delay=2):
    html_path = output_filename.replace('.png', '.html')
    folium_map.save(html_path)
    driver.get(f'file://{os.path.abspath(html_path)}')
    time.sleep(delay)
    driver.save_screenshot(output_filename)

In [6]:
import geopandas as gpd
import folium
import imageio
import os

# 1. Prepare GeoDataFrame for pilot cities
gdf_pilots = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
print("gdf_pilots columns:", gdf_pilots.columns) # Debug print

# 2. Determine earliest wave per province
# Spatial join to link cities to provinces. 'china' GeoDataFrame is assumed to be ready from previous cells.
cities_in_provinces = gpd.sjoin(china, gdf_pilots, how="inner", predicate='intersects')
print("cities_in_provinces columns:", cities_in_provinces.columns) # Debug print
print("cities_in_provinces head:\n", cities_in_provinces.head()) # Debug print

# Group by province name and find the earliest wave a city appeared in that province
# The column with city wave information after sjoin will be 'wave'
province_earliest_wave = cities_in_provinces.groupby('name_left').agg(earliest_wave=('wave', 'min')).reset_index()

# Merge this back with the original 'china' GeoDataFrame to get the geometries and wave info
china_with_waves = china.merge(province_earliest_wave, left_on='name', right_on='name_left', how='left')

# Drop the redundant 'name_left' column after merge
china_with_waves = china_with_waves.drop(columns=['name_left'])

# Fill NaN for provinces without pilot cities with 0, so they are not colored by a wave color.
# Convert to int type for consistency.
china_with_waves['earliest_wave'] = china_with_waves['earliest_wave'].fillna(0).astype(int)

# Create a dictionary for quick lookup of earliest_wave for each province name
province_wave_lookup = china_with_waves.set_index('name')['earliest_wave'].to_dict()

image_filenames = []

# Loop through waves to create and save maps
for current_wave in range(1, 5): # Iterate through the 4 waves
    # Create a new Folium map for each wave, centered on China
    m_wave = folium.Map(location=[32.0, 110.0], zoom_start=4, tiles='CartoDB positron')

    # Define the styling function for GeoJson features (provinces)
    def style_province_geojson(feature):
        province_name = feature['properties']['name']
        wave_num = province_wave_lookup.get(province_name, 0) # Default to 0 if no pilot city

        if wave_num > 0 and wave_num <= current_wave:
            # Color provinces that have pilot cities up to the current wave
            fill_color = wave_colors.get(wave_num, '#dddddd') # Use wave_colors for the identified wave
        else:
            # Light grey for non-pilot provinces or provinces whose pilot wave is in the future
            fill_color = '#dddddd'

        return {
            'fillColor': fill_color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }

    # Add the GeoJson layer with conditional styling for provinces to the map
    # china.__geo_interface__ provides the GeoJSON dictionary representation of the 'china' GeoDataFrame
    folium.GeoJson(
        china.__geo_interface__, # All of China's provincial boundaries
        style_function=style_province_geojson
    ).add_to(m_wave)

    # Add CircleMarkers for cities up to the current wave (to show exact city locations)
    df_current_wave_cities = df[df['wave'] <= current_wave]
    for _, row_city in df_current_wave_cities.iterrows():
        popup = f"<b>{row_city['name']}</b><br>Wave: {row_city['wave']}<br>Type: {row_city['type']}<br>{row_city['note']}"
        folium.CircleMarker(
            location=[row_city['lat'], row_city['lon']],
            radius=radius_map.get(row_city['type'], 8),
            color=wave_colors[row_city['wave']], fill=True, fillColor=wave_colors[row_city['wave']],
            fillOpacity=0.9, weight=2, popup=folium.Popup(popup, max_width=250)
        ).add_to(m_wave)

    # Add title overlay to the map
    title_html = f'''
         <h3 align="center" style="font-size:16px"><b>e-CNY Pilot Expansion - {wave_names[current_wave]}</b></h3>
         '''
    m_wave.get_root().html.add_child(folium.Element(title_html))

    # Save the map as a PNG image
    output_png = f'ecny_pilot_wave_{current_wave}.png'
    save_folium_map_as_png(m_wave, output_png)
    image_filenames.append(output_png)
    print(f"Generated {output_png}")

# Create GIF from the saved PNG images
output_gif = 'ecny_pilot_expansion.gif'
with imageio.get_writer(output_gif, mode='I', fps=1) as writer:
    for filename in image_filenames:
        image = imageio.v2.imread(filename)
        writer.append_data(image)

print(f"Animated GIF saved to {output_gif}")

# Clean up generated PNG and HTML files
for filename in image_filenames:
    os.remove(filename)
    os.remove(filename.replace('.png', '.html'))
print("Cleaned up temporary image and html files.")

# Quit the Selenium WebDriver
driver.quit()

gdf_pilots columns: Index(['name', 'lat', 'lon', 'wave', 'type', 'note', 'geometry'], dtype='object')
cities_in_provinces columns: Index(['featurecla', 'scalerank', 'adm1_code', 'diss_me', 'iso_3166_2',
       'wikipedia', 'iso_a2', 'adm0_sr', 'name_left', 'name_alt',
       ...
       'FCLASS_UA', 'FCLASS_TLC', 'geometry', 'index_right', 'name_right',
       'lat', 'lon', 'wave', 'type_right', 'note_right'],
      dtype='object', length=129)
cities_in_provinces head:
                     featurecla  scalerank adm1_code  diss_me iso_3166_2  \
249   Admin-1 states provinces          2  CHN-1810     1810      CN-YN   
249   Admin-1 states provinces          2  CHN-1810     1810      CN-YN   
960   Admin-1 states provinces          2  CHN-1813     1813      CN-LN   
1214  Admin-1 states provinces          2  CHN-1152     1152      CN-GX   
1619  Admin-1 states provinces          2  CHN-1180     1180      CN-GD   

     wikipedia iso_a2  adm0_sr  name_left                         name_alt 

# check GIF and HTML successfully saved